In [ ]:
#http://localhost:9200/hawker_reviews/_search?pretty
from pprint import pprint
from elasticsearch import Elasticsearch,helpers
es = Elasticsearch(['http://localhost:9200'], basic_auth=("elastic", "5vRl7G_wpBvo8CytUL=h"))
client_info = es.info()
print("Connected to ElasticSearch")
pprint(client_info.body)

In [ ]:
import json 
with open('reviews.json','r',encoding='utf-8') as f:
    reviews = json.load(f)
    unique_hawkers = set()
for review in reviews:
    hawker_name =  review.get('title','')
    if hawker_name:
        unique_hawkers.add(hawker_name)

print(f"Found {len(unique_hawkers)} unique hawkers,{unique_hawkers}")


In [ ]:
import  os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import requests
url ="https://www.onemap.gov.sg/api/auth/post/getToken"
payload = {
        "email":os.environ["ONEMAP_EMAIL"],
        "password": os.environ["ONEMAP_PASSWORD"]
    }

response=requests.request("POST",url,json=payload)

print(response.text)

In [ ]:
from urllib.parse import quote

def geocode_loc(query):
    encoded_query = quote(query,safe="&")
    url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={encoded_query}&returnGeom=Y&getAddrDetails=Y"
    # params = {"searchVal": query, "returnGeom": "Y","getAddrDetails":"Y"}
    headers = {"Authorization":os.environ["ONEMAP_TOKEN"]}
    response = requests.get(url,headers=headers)
    data = response.json()
    #print(f"Found: {data.get('found', 0)}")
   

    if data.get('found', 0) > 0:
        result = data['results'][0]
        return float(result['LATITUDE']), float(result['LONGITUDE'])

    print(data.get("error"))
    return None, None



geocode_loc("Geylang Serai Market and Food Centre ")

In [ ]:
hawker_dict = {}

manual_coords = {
    "Geylang East Market & Food Centre" : {"lat": 1.320681,"lon":103.886664},
    "Geylang Serai Market and Food Centre" : {"lat":1.316846 ,"lon":103.898282},
    "Whampoa Food Block 91 Food Centre" : {"lat": 1.323477,"lon":103.854081},
    "Bedok 85 Market" : {"lat":1.332117 ,"lon":103.9387361964381},
    "Hougang Avenue 1 Block 105 Market and Food Centre" : {"lat":1.354257 ,"lon":103.890022},
    "Promenade Market @ 84" : {"lat": 1.302430,"lon":103.906214},
}

for i, hawker_name in enumerate(unique_hawkers, 1):
    lat, lon = geocode_loc(hawker_name)
    if lat and lon:
        hawker_dict[hawker_name] ={"lat":lat, "lon":lon}
        print(f"{i}/{len(unique_hawkers)}: {hawker_name} ({lat:.4f}, {lon:.4f})")
    else:
        print(f"Failed: '{hawker_name}'")
        print(f"In manual_coords? {hawker_name in manual_coords}")

        if hawker_name in manual_coords:
            hawker_dict[hawker_name] = manual_coords[hawker_name]
        else:
            print(f"{hawker_name} lat and lon not found ")




In [ ]:
len(hawker_dict)

In [ ]:
for review in reviews:
    hawker_name = review.get("title","")
    if hawker_name in hawker_dict:
        review["location"] = hawker_dict[hawker_name]

with open("reviews_with_coords.json", "w",encoding="utf-8") as f:
  json.dump(reviews, f,indent=2, ensure_ascii =False)
print("Add location to reviews")

In [ ]:
#GET UNIQUE REVIEW CONTEXT KEYS
import json
from collections import Counter

with open("reviews_with_coords.json","r",encoding = "utf-8") as f:
 reviews = json.load(f)
all_fields = Counter()
for reviewObj in reviews:
    context =  reviewObj.get("reviewContext",{})
    all_fields.update(context.keys())

for field, count in all_fields.most_common():
    print(f"{field}: {count}")


In [ ]:
#GET UNIQUE REVIEW CONTEXT NUMERIC FIELD VALUES
import json
def uniqueVals(keyname):
  with open("reviews_with_coords.json","r",encoding = "utf-8") as f:
      reviews = json.load(f)
      all_vals  = set()
      for reviewObj in reviews:
       context =  reviewObj.get("reviewContext",{})
       numField = context.get(keyname)
       all_vals.add(numField )
      

      print(all_vals)

#uniqueVals("Price per person")
#uniqueVals("Wait time")
#uniqueVals("Group size")
for field in all_fields:
  print (field)
  uniqueVals(field)

In [ ]:
import re
def parse_price(text):
    if not text :
        return None,None
    nums = list(map(int, re.findall(r"\d+", text)))
    if not nums:
        return None, None

    elif "+" in text:
        return nums[0], None
    
    else: 
        return nums[0], nums[1]
    
print(parse_price("$10–20"))
print(parse_price("$100+"))
print(parse_price(None))

In [ ]:
def parse_wait_time(text):
     if not text:
        return None,None
     if "No wait" in text:
         return 0,0
     nums = list(map(int, re.findall(r"\d+", text)))
     if "hour" in text:
        mins = nums[0] * 60
        if "+" in text:
         return mins, None
        return mins,mins
     if "Up to" in text:
        return 0, nums[0]
     if len(nums) == 1:
        return nums[0], nums[0]
     if len(nums) >= 2:
        return nums[0], nums[1]
     return None,None

print(parse_wait_time("10-30 min"))
print(parse_wait_time("No wait"))
print(parse_wait_time("Up to 10 min"))
print(parse_wait_time("1+ hour"))
print(parse_wait_time(None))

In [ ]:
mapping = {
    "mappings":{
        "properties":{
            "hawker_name":{
                "type": "text",
                "fields": {"keyword": {"type": "keyword"}}
            },"location": {
                "type": "geo_point"
            },"review_text": {
                "type": "text",
                "analyzer": "standard"
            },
            "review_rating": {"type": "integer"},
            "review_author": {"type": "keyword"},
            "review_date": {"type": "date"},

            "food_rating": {"type": "integer"},
            "service_rating": {"type": "integer"},
            "atmosphere_rating": {"type": "integer"},
             

            "review_context": {"type": "flattened"},

            "context_wait_min": { "type": "integer" },
            "context_wait_max": { "type": "integer" },
            "context_price_min": { "type": "integer" },
            "context_price_max": { "type": "integer" }, 
            "context_parking_space": { "type": "keyword" },

            "context_recommended": {
               "type": "text",
               "analyzer": "english",
              "fields": {
                "raw": { "type": "keyword" }
                }
            },

              "context_meal_type": { "type": "keyword" },    
            }
    }
}

index_name = "hawker_reviews"
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
else:
    es.indices.create(index=index_name, body=mapping)




In [ ]:
import json
with open("reviews_with_coords.json","r",encoding = "utf-8") as f:
    reviews = json.load(f)
count = 0 
actions = []
for reviewObj in reviews :
    context = reviewObj.get('reviewContext', {})
    location = reviewObj.get('location') 
    detailed_rating = reviewObj.get('reviewDetailedRating', {})
    doc = {
            "hawker_name": reviewObj.get("title",""),
            "location": location,
            "review_text": reviewObj.get('text', ''),
            #"review_rating": reviewObj.get('stars', 0),
            "review_author":reviewObj.get('name', 'Anonymous'),
            "review_date": reviewObj.get('publishedAtDate', ''),
            
            
          
           
          
            "review_context": context if context else {}

            

    }

    if reviewObj.get("stars"):
       doc["review_rating"] = reviewObj.get("stars")
    if detailed_rating.get('Food'):
            doc["food_rating"] = detailed_rating['Food']
    if detailed_rating.get('Service'):
            doc["service_rating"] = detailed_rating['Service']
    if detailed_rating.get('Atmosphere'):
            doc["atmosphere_rating"] = detailed_rating['Atmosphere']

    
    #review context extraction 
    if context.get("Wait time"):
        doc["context_wait_min"], doc["context_wait_max"] = parse_wait_time(context.get("Wait time"))
    if context.get("Price per person"):
        doc["context_price_min"],doc["context_price_max"] = parse_price(context.get("Price per person"))

    if context.get("Parking space"):
         doc["context_parking_space"] =  context.get("Parking space")
    if context.get("Recommended dishes"):
         doc["context_recommended"] =  context.get("Recommended dishes")
    if context.get("Meal type"):
         doc["context_meal_type"] = context.get("Meal type")
   
    if doc['review_text']:
        actions.append({
            "_index": index_name,
            "_source": doc
        })
        count += 1
    
helpers.bulk(es, actions)
print(f"Indexed {count} reviews")      